# Operating Systems — Lecture 0: Why Does an OS Need Hardware Support?

## The Fundamental Question

Before studying processes, threads, virtual memory, or file systems, we should answer a more fundamental question:

**An operating system is itself a program. So how can it control every other program?**

Let's imagine we are computer engineers in the 1950s.

We have a CPU, RAM, and disks, but no modern operating system.

Our goal is simple:

> Allow three mutually untrusted programs — A, B, and C — to share one computer safely and efficiently, even if one program contains bugs or intentionally behaves maliciously.

For each hardware mechanism, we'll ask:

1. What problem are we trying to solve?
2. What happens if the hardware does not support it?
3. Why can't software alone reliably solve the problem?
4. What OS abstraction can we build using this hardware support?

The central idea is:

**Operating systems and CPU hardware co-evolved. Hardware did not provide a complete set of OS features from the beginning. Software requirements drove hardware innovations, and new hardware capabilities enabled more sophisticated operating systems.**

---

## 1. The Six Most Important Hardware Mechanisms

| Hardware Support | What Goes Wrong Without It? | What the OS Builds |
|---|---|---|
| **1. Privilege Levels** | Program A can execute privileged instructions and take control of the machine. | Kernel Mode / User Mode |
| **2. Trap / System Call Entry** | A cannot safely request privileged services. | System Calls |
| **3. Timer Interrupt** | A can enter an infinite loop and never voluntarily release the CPU. | Preemptive Multitasking |
| **4. Memory Protection** | A can overwrite B's or the kernel's memory. | Process Isolation |
| **5. Privileged I/O Access** | A can bypass the OS and directly manipulate hardware devices. | Device Protection |
| **6. Atomic Operations** | Concurrent modifications of shared data can produce race conditions. | Locks / Synchronization |

These are fundamental building blocks of modern general-purpose operating systems.

However, we must distinguish between two situations:

- Hardware capabilities that provide security guarantees software cannot enforce on unrestricted native code.
- Hardware capabilities that mainly make software solutions more efficient, practical, or easier to implement.

### 1.1 Privilege Levels — Preventing a Program from Taking Over

**The problem**

Suppose Program A has exactly the same hardware privileges as the kernel.

Then A could attempt:

```text
A: Disable memory protection
A: Read B's private memory
A: Modify OS control registers
```

How could the OS stop A if they have identical privileges?

It cannot reliably enforce a boundary against unrestricted native code with equal control over the machine.

**The hardware solution**

The CPU supports different privilege levels.

For example, modern x86 systems commonly use:

- Ring 0: Kernel Mode
- Ring 3: User Mode

The CPU rejects privileged operations attempted from User Mode.

**Key insight:** Privilege enforcement happens in hardware, not through a software `if` statement executed by the kernel.

### 1.2 Trap / System Call Entry — How Can a User Program Request Kernel Services?

**The problem**

Program A runs without hardware privileges, but sometimes it needs a privileged operation.

For example:

```text
A: Please read file.txt for me.
```

We need a way for A to request kernel services without giving A unrestricted kernel privileges.

**The hardware solution**

The CPU supports a controlled transition into a privileged handler.

```text
User Mode
    |
    | Trap / System Call
    v
Kernel Mode
    |
    | Execute trusted handler
    |
    | Return from trap
    v
User Mode
```

The distinction is important:

- **Hardware:** Provides a safe mechanism to enter privileged execution.
- **OS:** Defines which services exist and how they work.

For example, `read()`, `write()`, and `open()` are OS interfaces, not universal hardware instructions.

A CPU does not necessarily need an instruction literally named `SYSCALL`. Different architectures and older systems can use different trap mechanisms.

**Key insight: Hardware provides the mechanism; the OS defines the service.**

### 1.3 Timer Interrupt — Preventing a Program from Monopolizing the CPU

**The problem**

Suppose Program A executes:

```python
while True:
    pass
```

In a purely cooperative scheduling system, programs must voluntarily give up the CPU.

If A never yields, B and C may never execute.

**The hardware solution**

A timer generates an interrupt after a configured interval.

The CPU transfers control to a protected OS handler.

The OS can then:

1. Save A's execution state.
2. Select another runnable program.
3. Restore B's execution state.
4. Resume execution with B.

This enables **Preemptive Multitasking**.

```text
Time →

| Program A | OS | Program B | OS | Program A |
             ↑                 ↑
        Timer Interrupt   Timer Interrupt
```

The fundamental requirement is a trusted, asynchronous mechanism for regaining control of the CPU.

It does not necessarily require one particular kind of timer device.

**Key insight:** The OS no longer needs to trust programs to voluntarily surrender the CPU.

### 1.4 Memory Protection — Preventing Programs from Accessing Each Other's Memory

**The problem**

Suppose A contains a faulty pointer:

```text
Process A: Write to an address belonging to Process B.
```

Without memory protection, A may corrupt B's data or even kernel memory.

**The hardware solution**

The CPU checks whether memory accesses are permitted.

Possible mechanisms include:

- Base and Bounds Registers
- Segmentation
- MMU + Page Tables

Modern general-purpose operating systems commonly use page-based virtual memory.

Example:

| Virtual Page | Physical Page | Permission |
|---|---|---|
| Page A | Physical Page 7 | Read/Write |
| Page B | Physical Page 12 | Read Only |
| Page C | Unmapped | No Access |

The CPU checks the address translation and access permissions.

An invalid access causes a hardware exception, typically a Page Fault.

**Important:** Page tables are not the only possible memory-protection mechanism.

**Key insight:** The kernel configures memory-access rules; CPU hardware enforces them.

### 1.5 Privileged I/O — Preventing Programs from Directly Controlling Hardware

**The problem**

Imagine Program A can directly issue arbitrary disk commands:

```text
A: Overwrite disk blocks containing B's files.
```

Even perfect memory isolation would not prevent A from destroying B's data.

**The hardware solution**

Hardware restricts access to device-control operations through mechanisms such as:

- Privileged instructions
- I/O permission checks
- Protected memory-mapped device registers

A must request device operations through the OS.

The OS can then check permissions and coordinate access.

**Key insight:** Protecting RAM alone is not enough. Devices also require controlled access.

### 1.6 Atomic Operations — Building Reliable Synchronization

**The problem**

Suppose two execution threads share a counter:

```text
counter = 0

A reads 0
B reads 0
A writes 1
B writes 1
```

The final value is 1, not 2.

This is a **Race Condition**.

**The hardware solution**

CPUs provide atomic operations such as:

- Compare-and-Swap (CAS)
- Test-and-Set
- Atomic Fetch-and-Add

These enable software to implement synchronization mechanisms such as mutexes and locks.

However, there is an important qualification:

**Without CAS, mutual exclusion is not universally impossible.**

Under appropriate memory-model assumptions, software-only algorithms such as Peterson's algorithm and Lamport's Bakery algorithm can also provide mutual exclusion.

Hardware atomic instructions provide a practical, general-purpose foundation for modern concurrent systems.

**Key insight:** Hardware supplies synchronization primitives; the OS and applications build higher-level synchronization mechanisms.

---

## 2. Other Important Hardware Support

The first six mechanisms explain many of the fundamental problems. But modern systems depend on additional hardware capabilities.

| Hardware Support | What Happens Without It? | Can Software Compensate? |
|---|---|---|
| **7. Reset / Boot Mechanism** | The CPU does not know where to begin execution after power-on. | Some hardware-defined initial execution mechanism is necessary. |
| **8. Exceptions / Faults** | The OS cannot use normal exception-handling mechanisms to respond to invalid instructions or memory accesses. | Directly executed machine instructions need hardware support for reliable fault detection and reporting. |
| **9. CPU Registers / Execution State** | Programs cannot be correctly paused and resumed without preserving their execution state. | The OS can save and restore most state using software; a dedicated context-switch instruction is unnecessary. |
| **10. Device Interrupts** | The OS cannot receive asynchronous notification that an I/O operation has completed. | Yes. Polling is possible, although potentially inefficient. |
| **11. Cache Coherence / Memory Ordering** | Multiple CPUs may not observe shared-memory operations in the expected order. | Some systems can use software protocols, but shared-memory programming depends on defined consistency guarantees. |
| **12. DMA / IOMMU** | Data movement may require more CPU involvement, and DMA-capable devices may access memory without proper isolation. | DMA primarily improves performance. IOMMU provides hardware isolation for device memory access. |
| **13. Clock / Time Source** | The OS cannot reliably measure real elapsed time or schedule time-based operations. | An external time source can help; pure computation alone does not provide an independent wall-clock reference. |
| **14. Nonvolatile Storage** | Information stored only in volatile RAM disappears after power loss. | Persistent storage requires some form of nonvolatile medium. |

Not every OS requires all these capabilities.

A small embedded operating system may work without an MMU, disk, or DMA.

A modern general-purpose OS such as Linux relies heavily on many of them.

---

## 3. Hardware Mechanisms vs. OS Abstractions

This is perhaps the most important conceptual distinction.

### Hardware provides mechanisms

Examples:

- Privilege Levels
- Timer Interrupts
- Memory Protection
- Atomic Instructions
- Device Controllers

### The OS builds abstractions

Examples:

- System Calls
- CPU Scheduling
- Virtual Memory
- Mutexes and Semaphores
- Files, Pipes, and Sockets

A simplified mapping:

| Hardware Mechanism | Typical OS Abstraction |
|---|---|
| Privilege Levels + Controlled Traps | System Calls |
| Timer Interrupts + CPU State | Preemptive Scheduling |
| MMU + Memory Protection | Virtual Memory / Process Isolation |
| Atomic Operations + Memory Ordering | Mutexes / Semaphores |
| Device Controllers + Protected I/O | File and I/O Interfaces |

These relationships are not strictly one-to-one. Most OS abstractions combine multiple hardware mechanisms.

For example, **the CPU does not directly provide a Process abstraction.**

The CPU provides registers, execution state, interrupts, privilege levels, and memory-management mechanisms.

The OS combines these capabilities to create the illusion that each process owns an independent execution environment.

Likewise, the CPU does not inherently understand files, pipes, or sockets.

These are software abstractions built on lower-level mechanisms.

---

## 4. A Critical Distinction: Impossible Without Hardware vs. Possible but Inefficient

We should avoid treating every hardware feature as absolutely necessary.

**Hardware is most essential when the OS must enforce rules against software that cannot be trusted.**

Other mechanisms primarily improve performance, efficiency, and implementation complexity.

This explains the historical shift from cooperative systems toward hardware-supported protection and preemption.
